# Missão Aurora-1 — Verificação de Telemetria Pré-Lançamento

**Atividade Integradora — FIAP, Fase 1**

Este notebook executa o checklist de pré-lançamento do foguete Aurora-1: lê a
telemetria da janela T-10 min, aplica as faixas seguras de cada parâmetro,
decide entre `PRONTO PARA DECOLAR` e `DECOLAGEM ABORTADA`, e calcula a
autonomia energética da missão.

A lógica não é redefinida aqui: ela vive em `src/missao.py` e é coberta por
testes automatizados em `tests/test_missao.py`. O notebook importa esse módulo,
de modo que o que você executa abaixo é exatamente o código testado.

## 0. Preparação do ambiente

Se estiver no Google Colab, descomente e rode a célula de clone. Localmente,
basta executar a célula seguinte a partir do repositório.

In [1]:
# No Google Colab, descomente as duas linhas abaixo:
# !git clone https://github.com/SEU_USUARIO/SEU_REPOSITORIO.git
# %cd SEU_REPOSITORIO/entrega-fase1/notebooks

In [2]:
import sys
from pathlib import Path

# Localiza a raiz de entrega-fase1/ (quem tem src/missao.py) sem depender
# do nome da pasta atual, para funcionar a partir da raiz do notebook, de
# entrega-fase1/ ou da raiz do repositório.
cwd = Path.cwd()
candidatos = [cwd, cwd.parent, cwd / "entrega-fase1"]
RAIZ = next((c for c in candidatos if (c / "src" / "missao.py").exists()), None)
if RAIZ is None:
    raise RuntimeError(
        f"Não encontrei src/missao.py a partir de {cwd}. Rode o notebook "
        "de dentro do repositório (raiz, entrega-fase1/ ou "
        "entrega-fase1/notebooks/)."
    )
sys.path.insert(0, str(RAIZ / "src"))

import missao

print("Módulo carregado de:", missao.__file__)
print("Telemetria em:", missao.CAMINHO_PADRAO)

Módulo carregado de: /Users/gustavo.nakahodo/Nakahodo/projects/fiap/entrega-fase1/src/missao.py
Telemetria em: /Users/gustavo.nakahodo/Nakahodo/projects/fiap/entrega-fase1/data/telemetria.json


## 1. Organização e descrição da telemetria

Os dados chegam em `data/telemetria.json` com três cenários: o nominal e dois
de falha, usados para demonstrar o comportamento do algoritmo. Cada parâmetro
tem uma faixa segura definida como premissa do projeto.

In [3]:
dados = missao.carregar_telemetria(missao.CAMINHO_PADRAO, "nominal")

print(f"{'Parâmetro':<28} {'Leitura':>12}  {'Faixa segura':<22} Situação")
print("-" * 78)
for chave, (minimo, maximo) in missao.FAIXAS.items():
    valor = dados[chave]
    unidade = missao.UNIDADES[chave]
    situacao = "OK" if minimo <= valor <= maximo else "FORA DA FAIXA"
    print(f"{missao.ROTULOS[chave]:<28} {f'{valor} {unidade}':>12}  "
          f"{f'{minimo} a {maximo} {unidade}':<22} {situacao}")

print()
print(f"{'Integridade estrutural':<28} {dados['integridade_estrutural']:>12}  "
      f"{'= 1':<22} {'OK' if dados['integridade_estrutural'] == 1 else 'COMPROMETIDA'}")

valor_energia = f"{dados['nivel_energia_pct']} %"
situacao_energia = ("OK" if dados['nivel_energia_pct'] >= missao.ENERGIA_MINIMA_PCT
                     else "INSUFICIENTE")
print(f"{'Nível de energia':<28} {valor_energia:>12}  "
      f"{'>= 85.0 %':<22} {situacao_energia}")

print()
print("Módulos críticos:")
for nome in missao.MODULOS_CRITICOS:
    print(f"  - {missao.ROTULOS[nome]:<22} {dados['modulos_criticos'][nome]}")

Parâmetro                         Leitura  Faixa segura           Situação
------------------------------------------------------------------------------
Temperatura interna               23.4 °C  18.0 a 28.0 °C         OK
Temperatura externa               12.8 °C  -40.0 a 50.0 °C        OK
Pressão do tanque de LOX        228.0 bar  200.0 a 250.0 bar      OK
Pressão do tanque de RP-1       214.0 bar  190.0 a 240.0 bar      OK

Integridade estrutural                  1  = 1                    OK
Nível de energia                   92.0 %  >= 85.0 %              OK

Módulos críticos:
  - navegação              OK
  - comunicação            OK
  - propulsão              OK
  - controle térmico       OK
  - suporte de vida        OK


## 2. Algoritmo de verificação

O fluxograma está em [`docs/fluxograma.md`](../docs/fluxograma.md) e o
pseudocódigo em [`docs/pseudocodigo.md`](../docs/pseudocodigo.md).

Duas decisões governam o algoritmo:

1. **Limites inclusivos.** Uma leitura de exatamente 28,0 °C é aprovada.
2. **Nenhuma parada antecipada.** Todas as verificações são executadas mesmo
   depois da primeira reprovação, para que o operador receba a lista completa
   de problemas de uma só vez.

In [4]:
resultado = missao.verificar_telemetria(dados)

print("Decisão:", resultado.decisao)
print("Aprovado:", resultado.aprovado)
print("Falhas:", resultado.falhas or "(nenhuma)")

Decisão: PRONTO PARA DECOLAR
Aprovado: True
Falhas: (nenhuma)


## 3. Script em Python — execução completa

A função `executar` percorre o fluxo inteiro: leitura dos dados, execução das
verificações e impressão do resultado final.

In [5]:
print(missao.executar("nominal"))

RELATÓRIO DE PRÉ-LANÇAMENTO — MISSÃO Aurora-1
Cenário de telemetria: nominal

1. TELEMETRIA
  Parâmetro                       Leitura  Faixa segura         Situação
  --------------------------------------------------------------------------
  Temperatura interna             23.4 °C  18.0 a 28.0 °C       OK
  Temperatura externa             12.8 °C  -40.0 a 50.0 °C      OK
  Pressão do tanque de LOX      228.0 bar  200.0 a 250.0 bar    OK
  Pressão do tanque de RP-1     214.0 bar  190.0 a 240.0 bar    OK
  Integridade estrutural                1  = 1                  OK
  Nível de energia                 92.0 %  >= 85.0 %            OK

  Módulos críticos:
    - navegação              OK
    - comunicação            OK
    - propulsão              OK
    - controle térmico       OK
    - suporte de vida        OK

2. ANÁLISE ENERGÉTICA
  Capacidade total .................. 120.00 kWh
  Carga atual ....................... 92.00 %
  Perdas energéticas ................ 8.00 %
  Energia di

### 3.1 Cenários de falha

Para demonstrar que o algoritmo aborta corretamente — e reporta *todos* os
motivos, não apenas o primeiro.

In [6]:
print(missao.executar("falha_termica"))

RELATÓRIO DE PRÉ-LANÇAMENTO — MISSÃO Aurora-1
Cenário de telemetria: falha_termica

1. TELEMETRIA
  Parâmetro                       Leitura  Faixa segura         Situação
  --------------------------------------------------------------------------
  Temperatura interna             31.2 °C  18.0 a 28.0 °C       FORA DA FAIXA
  Temperatura externa             12.8 °C  -40.0 a 50.0 °C      OK
  Pressão do tanque de LOX      228.0 bar  200.0 a 250.0 bar    OK
  Pressão do tanque de RP-1     214.0 bar  190.0 a 240.0 bar    OK
  Integridade estrutural                1  = 1                  OK
  Nível de energia                 92.0 %  >= 85.0 %            OK

  Módulos críticos:
    - navegação              OK
    - comunicação            OK
    - propulsão              OK
    - controle térmico       OK
    - suporte de vida        OK

2. ANÁLISE ENERGÉTICA
  Capacidade total .................. 120.00 kWh
  Carga atual ....................... 92.00 %
  Perdas energéticas ................ 8.

In [7]:
print(missao.executar("falha_multipla"))

RELATÓRIO DE PRÉ-LANÇAMENTO — MISSÃO Aurora-1
Cenário de telemetria: falha_multipla

1. TELEMETRIA
  Parâmetro                       Leitura  Faixa segura         Situação
  --------------------------------------------------------------------------
  Temperatura interna             34.5 °C  18.0 a 28.0 °C       FORA DA FAIXA
  Temperatura externa             12.8 °C  -40.0 a 50.0 °C      OK
  Pressão do tanque de LOX      188.0 bar  200.0 a 250.0 bar    FORA DA FAIXA
  Pressão do tanque de RP-1     214.0 bar  190.0 a 240.0 bar    OK
  Integridade estrutural                0  = 1                  COMPROMETIDA
  Nível de energia                 78.0 %  >= 85.0 %            INSUFICIENTE

  Módulos críticos:
    - navegação              OK
    - comunicação            FALHA
    - propulsão              OK
    - controle térmico       FALHA
    - suporte de vida        OK

2. ANÁLISE ENERGÉTICA
  Capacidade total .................. 120.00 kWh
  Carga atual ....................... 78.00 %
  

## 4. Análise energética

$$E_{disp} = C_{total} \times \frac{carga}{100} \times \left(1 - \frac{perdas}{100}\right)$$

$$E_{rest} = E_{disp} - E_{decolagem} \qquad
  t_{autonomia} = \frac{E_{rest}}{P_{voo}} \qquad
  margem = \frac{E_{rest}}{E_{decolagem}} \times 100$$

In [8]:
energia = missao.analisar_energia(dados["nivel_energia_pct"])

print(f"Capacidade total ......... {missao.CAPACIDADE_TOTAL_KWH:>8.2f} kWh")
print(f"Carga atual .............. {dados['nivel_energia_pct']:>8.2f} %")
print(f"Perdas energéticas ....... {missao.PERDAS_PCT:>8.2f} %")
print(f"Energia disponível ....... {energia.disponivel_kwh:>8.2f} kWh")
print(f"Consumo na decolagem ..... {missao.CONSUMO_DECOLAGEM_KWH:>8.2f} kWh")
print(f"Energia restante ......... {energia.restante_kwh:>8.2f} kWh")
print(f"Consumo em voo ........... {missao.CONSUMO_VOO_KW:>8.2f} kW")
print(f"Autonomia estimada ....... {energia.autonomia_h:>8.2f} h")
print(f"Margem sobre a decolagem . {energia.margem_pct:>8.2f} % "
      f"(mínimo {missao.MARGEM_MINIMA_PCT:.0f} %)")
print()
print("Parecer energético:", "ADEQUADO" if energia.aprovado else "INSUFICIENTE")

Capacidade total .........   120.00 kWh
Carga atual ..............    92.00 %
Perdas energéticas .......     8.00 %
Energia disponível .......   101.57 kWh
Consumo na decolagem .....    45.00 kWh
Energia restante .........    56.57 kWh
Consumo em voo ...........     6.50 kW
Autonomia estimada .......     8.70 h
Margem sobre a decolagem .   125.71 % (mínimo 20 %)

Parecer energético: ADEQUADO


### 4.1 Sensibilidade da autonomia ao nível de carga

Qual é a carga mínima que ainda permite decolar com a margem de 20 %?

In [9]:
print(f"{'Carga (%)':>10} {'Disponível (kWh)':>18} {'Autonomia (h)':>15} "
      f"{'Margem (%)':>12}  Parecer")
print("-" * 70)
for carga in range(50, 101, 5):
    e = missao.analisar_energia(float(carga))
    parecer = "adequado" if e.aprovado else "insuficiente"
    print(f"{carga:>10} {e.disponivel_kwh:>18.2f} {e.autonomia_h:>15.2f} "
          f"{e.margem_pct:>12.2f}  {parecer}")

 Carga (%)   Disponível (kWh)   Autonomia (h)   Margem (%)  Parecer
----------------------------------------------------------------------
        50              55.20            1.57        22.67  adequado
        55              60.72            2.42        34.93  adequado
        60              66.24            3.27        47.20  adequado
        65              71.76            4.12        59.47  adequado
        70              77.28            4.97        71.73  adequado
        75              82.80            5.82        84.00  adequado
        80              88.32            6.66        96.27  adequado
        85              93.84            7.51       108.53  adequado
        90              99.36            8.36       120.80  adequado
        95             104.88            9.21       133.07  adequado
       100             110.40           10.06       145.33  adequado


## 5. Análise assistida por IA

A telemetria e as faixas seguras foram submetidas a um modelo de linguagem
(Claude), com o prompt reproduzido abaixo. A resposta e o comentário crítico
sobre ela estão no relatório, em
[`docs/relatorio.md`](../docs/relatorio.md#5-análise-assistida-por-ia).

> **Prompt enviado:**
> "Você é um engenheiro de sistemas de lançamento. A seguir está a telemetria
> de um foguete na janela T-10 min e as faixas seguras adotadas pela equipe.
> (1) Classifique cada parâmetro como nominal, de atenção ou crítico.
> (2) Identifique possíveis anomalias, inclusive correlações entre parâmetros
> que isoladamente pareceriam aceitáveis. (3) Liste os riscos de missão que
> essas leituras sugerem, do mais provável ao menos provável. Seja explícito
> sobre o que não é possível concluir apenas com esses dados."

A resposta do modelo foi avaliada criticamente — não aceita como veredito.
O ponto principal: a IA levanta hipóteses úteis sobre *correlações* que o
checklist determinístico não captura, mas não pode substituir o checklist,
porque não tem garantia de reprodutibilidade nem rastreabilidade de decisão.

## 6. Reflexão crítica

O texto completo sobre ética e responsabilidade, impacto social da exploração
espacial e sustentabilidade tecnológica está em
[`docs/relatorio.md`](../docs/relatorio.md#6-reflexão-crítica).

## 7. Testes automatizados

A lógica usada acima é coberta por testes. Para executá-los, a partir da raiz
do repositório:

```bash
pytest entrega-fase1/tests/ -v
```